In [2]:
!pip install -q pyvi
!pip install -q pycocoevalcap
!pip install -q pyvi


In [3]:
import json
from collections import defaultdict

VINTERN_FILE = 'vintern_inference_results.json'
METADATA_FILE = 'metadata.jsonl'
QWEN_FILE    = 'results_qwen2vl_test.jsonl'
BLIP2_FILE   = 'results_blip2_full.json'
OUTPUT_MERGED = 'comparison_results.json'

# ── 1. Đọc metadata → thứ tự file_name + captions ──
seen_order = []
file_to_captions = defaultdict(list)
with open(METADATA_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        fn = item['file_name']
        if fn not in file_to_captions:
            seen_order.append(fn)
        file_to_captions[fn].append(item['caption'])

# ── 2. Đọc Vintern → gán file_name theo index ──
with open(VINTERN_FILE, 'r', encoding='utf-8') as f:
    vintern_raw = json.load(f)
vintern_list = vintern_raw['predictions']

vintern_by_fn = {}
for i, item in enumerate(vintern_list):
    fn = seen_order[i]
    vintern_by_fn[fn] = item['prediction']

# ── 3. Đọc Qwen → giữ cả list (để lấy thứ tự) lẫn dict (để tra cứu) ──
qwen_entries = []   # ← list theo thứ tự gốc
qwen_by_fn   = {}
with open(QWEN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        qwen_entries.append(item)
        qwen_by_fn[item['file_name']] = item['prediction']

# ── 4. Đọc BLIP2 → gán file_name theo thứ tự Qwen (thứ tự khớp nhau) ──
with open(BLIP2_FILE, 'r', encoding='utf-8') as f:
    blip2_raw = json.load(f)
blip2_list = blip2_raw['predictions']   # list có "image_id" + "prediction"

blip2_by_fn = {}
for i, item in enumerate(blip2_list):
    if i < len(qwen_entries):
        fn = qwen_entries[i]['file_name']   # map theo cùng vị trí với Qwen
        blip2_by_fn[fn] = item['prediction']

# ── 5. Ghép dựa trên file_name ──
merged_data  = []
missing_qwen = []

for i, fn in enumerate(seen_order):
    if fn not in qwen_by_fn:
        missing_qwen.append(fn)
        continue
    merged_data.append({
        "image_id":    i,
        "file_name":   fn,
        "references":  file_to_captions[fn],
        "vintern_pred": vintern_by_fn[fn],
        "qwen2vl_pred": qwen_by_fn[fn],
        "blip2_pred":   blip2_by_fn.get(fn, None)
    })

# ── 6. Sắp xếp theo tên file (tăng dần) + cấp lại image_id ──
merged_data.sort(key=lambda x: x['file_name'])
for i, record in enumerate(merged_data):
    record['image_id'] = i
# Đảm bảo image_id là field đầu tiên cho dễ đọc
merged_data = [
    {"image_id": r.pop("image_id"), **r}
    for r in merged_data
]

# ── 7. Lưu kết quả ──
with open(OUTPUT_MERGED, 'w', encoding='utf-8') as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=4)

# ── 8. Báo cáo ──
print(f"✅ Đã ghép xong: {len(merged_data)} ảnh")
print(f"📁 Lưu tại: {OUTPUT_MERGED}")

blip2_missing = sum(1 for r in merged_data if r['blip2_pred'] is None)
if blip2_missing:
    print(f"⚠️  {blip2_missing} ảnh thiếu BLIP2 prediction")
else:
    print("✅ Tất cả ảnh đều có đủ BLIP2 prediction!")

if missing_qwen:
    print(f"⚠️  {len(missing_qwen)} ảnh thiếu trong Qwen:")
    for fn in missing_qwen[:10]:
        print(f"   - {fn}")
else:
    print("✅ Tất cả 800 ảnh có đủ trong cả 4 nguồn!")

# ── 9. Xem mẫu kiểm tra ──
print("\n📋 Mẫu record đầu tiên:")
print(json.dumps(merged_data[0], ensure_ascii=False, indent=2))


✅ Đã ghép xong: 800 ảnh
📁 Lưu tại: comparison_results.json
✅ Tất cả ảnh đều có đủ BLIP2 prediction!
✅ Tất cả 800 ảnh có đủ trong cả 4 nguồn!

📋 Mẫu record đầu tiên:
{
  "image_id": 0,
  "file_name": "00003.jpg",
  "references": [
    "con đường phía trước thẳng tắp và vắng phương tiện qua lại với biển báo giao thông nằm ở lề đất bên phải cạnh rừng thông, bạn hãy đi sát mép trái để tránh vật cản ven đường",
    "lòng đường trải nhựa rộng rãi được bao quanh bởi hàng cây thông cao vút hai bên và hiện tại không có xe cộ lưu thông, bạn có thể tự tin bước đi dọc theo sát mép đường bằng phẳng",
    "phía trước là đoạn đường trống trải có biển cảnh báo giao thông bên lề phải và cỏ mọc um tùm sát hai bên mép đường, bạn nên đi chậm trên phần đường nhựa để không vấp phải cỏ rậm",
    "không gian khu vực này rất thoáng đãng với dải phân cách làn đường rõ ràng nhưng không có vỉa hè dành cho người đi bộ, hãy luôn chú ý đi sát lề bên trái để đảm bảo sự an toàn",
    "tuyến đường êm ái băng qua khu vự

In [4]:
import json
import os
import pandas as pd
from pyvi import ViTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge

# --- HÀM CHẤM ĐIỂM CHUẨN TIẾNG VIỆT ---
def calculate_metrics(predictions, model_name):
    print(f'📝 Đang chấm điểm cho {model_name}...')
    gts = {}
    res = {}
    for item in predictions:
        if not item.get(model_name):
            continue
        idx = str(item['image_id'])
        pred_seg = ViTokenizer.tokenize(item[model_name].lower())
        res[idx] = [pred_seg]
        refs_seg = [ViTokenizer.tokenize(ref.lower()) for ref in item['references']]
        gts[idx] = refs_seg

    scorers = [
        (Bleu(4), ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']),
        (Cider(), 'CIDEr'),
        (Rouge(), 'ROUGE-L')
    ]
    final_scores = {}
    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        if isinstance(method, list):
            for m, s in zip(method, score):
                final_scores[m] = round(s * 100, 2)
        else:
            final_scores[method] = round(score * 100, 2)
    return final_scores

# --- THỰC THI CHẤM ĐIỂM ---
scores_vintern = calculate_metrics(merged_data, 'vintern_pred')
scores_qwen    = calculate_metrics(merged_data, 'qwen2vl_pred')
scores_blip2   = calculate_metrics(merged_data, 'blip2_pred')

# --- TỔNG HỢP VÀ IN BẢNG SO SÁNH ---
comparison_table = {
    'Vintern-1B-v3_5': {
        **scores_vintern,
        'Time/Img (s)': vintern_raw['system_metrics']['time_per_img_sec'],
        'VRAM (GB)':    vintern_raw['system_metrics']['peak_vram_GB'],
        'Params (M)':   vintern_raw['system_metrics']['params_M'],
        'Disk (GB)':    vintern_raw['system_metrics']['disk_size_GB'],   # ← thêm
    },
    'Qwen2-VL-2B': {
        **scores_qwen,
        'Time/Img (s)': 2.8127,
        'VRAM (GB)':    4.23,
        'Params (M)':   2210,
        'Disk (GB)':    4.13,                                            # ← thêm
    },
    'BLIP-2': {
        **scores_blip2,
        'Time/Img (s)': blip2_raw['system_metrics']['time_per_img_sec'],
        'VRAM (GB)':    blip2_raw['system_metrics']['peak_vram_GB'],
        'Params (M)':   blip2_raw['system_metrics']['params_M'],
        'Disk (GB)':    blip2_raw['system_metrics']['disk_size_GB'],     # ← thêm
    }
}

df = pd.DataFrame(comparison_table).T
print('\n' + '='*80)
print('🏆 BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG')
print('='*80)
print(df.to_string())
print('='*80)


📝 Đang chấm điểm cho vintern_pred...
{'testlen': 37590, 'reflen': 27628, 'guess': [37590, 36790, 35990, 35190], 'correct': [13716, 2496, 643, 200]}
ratio: 1.3605762270160213
📝 Đang chấm điểm cho qwen2vl_pred...
{'testlen': 31694, 'reflen': 26892, 'guess': [31694, 30894, 30094, 29294], 'correct': [12026, 2210, 425, 73]}
ratio: 1.1785661163170764
📝 Đang chấm điểm cho blip2_pred...
{'testlen': 18967, 'reflen': 23745, 'guess': [18967, 18167, 17367, 16567], 'correct': [5818, 986, 311, 77]}
ratio: 0.7987786902505454

🏆 BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG
                 BLEU-1  BLEU-2  BLEU-3  BLEU-4  CIDEr  ROUGE-L  Time/Img (s)  VRAM (GB)  Params (M)  Disk (GB)
Vintern-1B-v3_5   36.49   15.73    7.62    3.98   3.89    17.66        4.4005       2.30      938.19       3.50
Qwen2-VL-2B       37.94   16.48    7.26    3.13   4.52    17.66        2.8127       4.23     2210.00       4.13
BLIP-2            23.84   10.03    5.19    2.67   4.52    13.56        5.5797       4.18     3942.45      14.69


In [5]:
!pip install sentence-transformers


In [6]:
# --- Bảng 1: Chất lượng Caption ---
quality_cols = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'CIDEr', 'ROUGE-L']
df_quality = df[quality_cols]

print('=' * 70)
print('BẢNG 1 — CHẤT LƯỢNG CAPTION (Quality Metrics)')
print('=' * 70)
print(df_quality.to_string())
print()

# --- Bảng 2: Hiệu quả Tài nguyên ---
efficiency_cols = ['Params (M)', 'Disk (GB)', 'VRAM (GB)', 'Time/Img (s)']
df_efficiency = df[efficiency_cols]

print('=' * 70)
print('BẢNG 2 — HIỆU QUẢ TÀI NGUYÊN (Efficiency Metrics)')
print('=' * 70)
print(df_efficiency.to_string())


BẢNG 1 — CHẤT LƯỢNG CAPTION (Quality Metrics)
                 BLEU-1  BLEU-2  BLEU-3  BLEU-4  CIDEr  ROUGE-L
Vintern-1B-v3_5   36.49   15.73    7.62    3.98   3.89    17.66
Qwen2-VL-2B       37.94   16.48    7.26    3.13   4.52    17.66
BLIP-2            23.84   10.03    5.19    2.67   4.52    13.56

BẢNG 2 — HIỆU QUẢ TÀI NGUYÊN (Efficiency Metrics)
                 Params (M)  Disk (GB)  VRAM (GB)  Time/Img (s)
Vintern-1B-v3_5      938.19       3.50       2.30        4.4005
Qwen2-VL-2B         2210.00       4.13       4.23        2.8127
BLIP-2              3942.45      14.69       4.18        5.5797
